## Nvidia Stock Price Prediction - **LSTM**
Ethan P

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import sympy
import numpy as np
import matplotlib.pyplot as plt
import pdb

In [ ]:
# Read in the dataset #

df = pd.read_csv('nvidia_historical_stock_data.csv')
df = df[['Date','Close']]

device ='cuda:0' if torch.cuda.is_available() else 'cpu'
device

df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
# Format dataset #

from copy import deepcopy as dc

def prepare_dataframe_for_lstm(DataFrame, n_steps):
  DataFrame = dc(DataFrame)

  DataFrame['Date'] = pd.to_datetime(DataFrame['Date'])
  DataFrame.set_index('Date', inplace=True)

  for i in range(1, n_steps+1):
    DataFrame[f'Close-{i}'] = DataFrame['Close'].shift(i)

  DataFrame.dropna(inplace=True)
  return DataFrame


In [ ]:
# lookback columns to make it easier to find previous prices #

lookback = 7
shifted_DataFrame = prepare_dataframe_for_lstm(df, lookback)
shifted_DataFrame

shifted_DataFrame_as_np = shifted_DataFrame.to_numpy()
shifted_DataFrame_as_np


In [ ]:
# FEATURE SCALING - prevents disproportionate calculations #

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(-1,1))
shifted_DataFrame_as_np = scaler.fit_transform(shifted_DataFrame_as_np)
shifted_DataFrame_as_np

X = shifted_DataFrame_as_np[:, 1:]
y = shifted_DataFrame_as_np[:, 0]

X.shape, y.shape

X = dc(np.flip(X, axis=1))

split_index = int(len(X) * 0.9)
split_index

shifted_DataFrame_as_np

In [ ]:
# splits dataset into train and test #

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [ ]:
# reshapes dataset into that required by PyTorch #

X_train = X_train.reshape((-1, lookback, 1))
X_test = X_test.reshape((-1, lookback, 1))

y_train = y_train.reshape((-1, 1))
y_test = y_test.reshape((-1, 1))

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# Tensor format #

X_train = torch.tensor(X_train).float()
y_train = torch.tensor(y_train).float()

X_test = torch.tensor(X_test).float()
y_test = torch.tensor(y_test).float()

X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [ ]:
# Timeseries dataset --> effeciency... #

from torch.utils.data import Dataset

class TimeSeriesDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]

train_dataset = TimeSeriesDataset(X_train, y_train)
test_dataset = TimeSeriesDataset(X_test, y_test)


In [ ]:
from torch.utils.data import DataLoader

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

for _, batch in enumerate(train_loader):
  x_batch, y_batch = batch[0].to(device), batch[1].to(device)
  print(x_batch.shape, y_batch.shape)
  break


In [ ]:
class LSTM(nn.Module):

  def __init__(self, input_size, hidden_size, num_stacked_layers):
    super().__init__()
    self.hidden_size = hidden_size
    self.num_stacked_layers = num_stacked_layers
    self.lstm = nn.LSTM(input_size, hidden_size, num_stacked_layers, batch_first=True)
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
    batch_size = x.size(0)
    h0 = torch.zeros(self.num_stacked_layers, batch_size, self.hidden_size).to(device)
    c0 = torch.zeros(self.num_stacked_layers, batch_size, self.hidden_size).to(device)
    out, _ = self.lstm(x, (h0, c0))
    out = self.fc(out[:, -1, :])
    return out


In [ ]:

from pdb import run
def train_one_epoch():
  model.train(True)
  print(f'Epoch: {epoch + 1}')
  running_loss = 0.0

  for batch_index, batch in enumerate(train_loader):
    x_batch, y_batch = batch[0].to(device), batch[1].to(device)

    output = model(x_batch)
    loss = loss_function(output, y_batch)
    running_loss += loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if batch_index % 100 == 99: # print every 100 batches
      avg_loss_across_batches = running_loss / 100
      print('Batch {0}, loss: {1:.3f}'.format(batch_index + 1, avg_loss_across_batches))
      running_loss = 0.0

  print()


In [ ]:
def validate_one_epoch():
  model.train(False)
  running_loss = 0

  for batch_index, batch in enumerate(test_loader):
    x_batch, y_batch = batch[0].to(device), batch[1].to(device)

    with torch.no_grad():
      output = model(x_batch)
      loss = loss_function(output, y_batch)
      running_loss += loss

  avg_loss_across_batches = running_loss / len(test_loader)

  print('Val Loss: {0:.3f}'.format(avg_loss_across_batches))
  print('------------------------------------------------')
  print()
  return avg_loss_across_batches

model = LSTM(1, 256, 3)
model.to(device)
model

In [ ]:
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
num_epochs = 32
loss_function = nn.MSELoss()

validation_losses = [] # Initialize a list to store validation losses
best_val_loss = float('inf') # Initialize best validation loss to infinity

for epoch in range(num_epochs):
  train_one_epoch()
  val_loss = validate_one_epoch()
  validation_losses.append(val_loss.item()) # Store the validation loss

  # Save the model if the current validation loss is the best so far
  if val_loss < best_val_loss:
    best_val_loss = val_loss
    print(f'New best validation loss: {best_val_loss:.3f}. Saving model.')
    torch.save(model.state_dict(), 'best_model.pth') # Save the model state

with torch.no_grad():
  predicted = model(X_train.to(device)).to('cpu').numpy()

In [ ]:
import matplotlib.pyplot as plt

# Plot the validation losses stored during training
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(validation_losses) + 1), validation_losses, marker='o', linestyle='-', color='skyblue')
plt.title('Validation Loss Over Epochs (Dynamic)')
plt.xlabel('Epoch')
plt.ylabel('Validation Loss')
plt.grid(True)
plt.show()

In [ ]:
# Get actual datetime index for y_train
train_dates = shifted_DataFrame.index[:len(y_train)]

# Create dummy values to pad for inverse scaling
dummy_input = np.zeros((len(predicted), lookback))

# Rebuild full input for inverse scaling
recombined_actual = np.concatenate((y_train.numpy(), dummy_input), axis=1)
recombined_pred = np.concatenate((predicted, dummy_input), axis=1)

# Inverse transform using same scaler
y_train_unscaled = scaler.inverse_transform(recombined_actual)[:, 0]
predicted_unscaled = scaler.inverse_transform(recombined_pred)[:, 0]

# Plot results properly
plt.figure(figsize=(10, 6))
plt.plot(train_dates, y_train_unscaled, label='Actual Close', color='orange')
plt.plot(train_dates, predicted_unscaled, label='Predicted Close', color='green')
plt.xlabel('Date')
plt.ylabel('Close')
plt.title('LSTM Predicted vs Actual Close Prices')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

train_predictions = predicted.flatten()

dummies = np.zeros((X_train.shape[0], lookback+1))
dummies[:, 0] = train_predictions
dummies = scaler.inverse_transform(dummies)

train_predictions = dc(dummies[:, 0])
train_predictions

dummies = np.zeros((X_train.shape[0], lookback+1))
dummies[:, 0] = y_train.flatten()
dummies = scaler.inverse_transform(dummies)

new_y_train = dc(dummies[:, 0])
new_y_train


In [ ]:
'''plt.plot(new_y_train, label='Actual Close')
plt.plot(train_predictions, label='Predicted Close')
plt.xlabel('Close')
plt.ylabel('Day')
plt.legend()
plt.show()'''


In [ ]:
test_predictons = model(X_test.to(device)).detach().cpu().numpy().flatten()

dummies = np.zeros((X_test.shape[0], lookback+1))
dummies[:, 0] = test_predictons
dummies = scaler.inverse_transform(dummies)

test_predictions = dc(dummies[:, 0])
test_predictions


In [ ]:
test_predictons = model(X_test.to(device)).detach().cpu().numpy().flatten()

dummies = np.zeros((X_test.shape[0], lookback+1))
dummies[:, 0] = y_test.flatten()
dummies = scaler.inverse_transform(dummies)

new_y_test = dc(dummies[:, 0])
new_y_test


In [ ]:
plt.plot(new_y_test, label='Actual Close')
plt.plot(test_predictions, label='Predicted Close')
plt.xlabel('Day')
plt.ylabel('Close')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(new_y_test, test_predictions)
print(f'R² Accuracy-like Score: {r2:.4f}')

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Evaluate Training Set
print('Training Set Metrics:')
mae_train = mean_absolute_error(new_y_train, train_predictions)
mse_train = mean_squared_error(new_y_train, train_predictions)
rmse_train = np.sqrt(mse_train)
print(f'Mean Absolute Error (MAE): {mae_train:.4f}')
print(f'Mean Squared Error (MSE): {mse_train:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_train:.4f}')

print('\nTest Set Metrics:')
# Evaluate Test Set
mae_test = mean_absolute_error(new_y_test, test_predictions)
mse_test = mean_squared_error(new_y_test, test_predictions)
rmse_test = np.sqrt(mse_test)
print(f'Mean Absolute Error (MAE): {mae_test:.4f}')
print(f'Mean Squared Error (MSE): {mse_test:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_test:.4f}')

In [ ]:
def predict_stock_on_future_date(target_date_str):
    target_date = pd.to_datetime(target_date_str)
    last_known_date = df['Date'].max()

    if target_date <= last_known_date:
        print("That date is already in the dataset. Use `predict_stock_on_date()` instead.")
        return

    # Count how many business days between last known and target
    future_dates = pd.bdate_range(start=last_known_date + pd.Timedelta(days=1), end=target_date)
    n_days = len(future_dates)

    if n_days == 0:
        print("Target date is a non-business day or too close.")
        return

    # Start with last lookback window
    last_window = df['Close'].values[-lookback:]
    for _ in range(n_days):
        window = np.array(last_window[::-1]).reshape(1, lookback)
        dummy_input = np.zeros((1, lookback + 1))
        dummy_input[0, 1:] = window

        scaled_window = scaler.transform(dummy_input)[0, 1:].reshape(1, lookback, 1)
        x = torch.tensor(scaled_window).float().to(device)

        model.eval()
        with torch.no_grad():
            pred = model(x).cpu().numpy().flatten()

        dummy_output = np.zeros((1, lookback + 1))
        dummy_output[0, 0] = pred.item()
        predicted_price = scaler.inverse_transform(dummy_output)[0, 0]

        # Update lookback window
        last_window = np.append(last_window[1:], predicted_price)

    print(f"Predicted Close on {target_date_str}: ${predicted_price:.2f}")
    return predicted_price


In [ ]:
predict_stock_on_future_date('2025-05-31')

### Predict Future Stock Prices
You can use the `predict_stock_on_future_date` function to predict the closing price for a specific future date. Remember that this function iteratively predicts one day at a time, using its own predictions as input for the next day, which can accumulate errors over longer forecasts. The further into the future you predict, the less reliable the prediction might be.

### Further Model Exploration and Improvement

Here are some other things you can do with this model:

1.  **Hyperparameter Tuning**: Experiment with different `learning_rate`, `num_epochs`, `hidden_size`, `num_stacked_layers`, and `lookback` values to find an optimal configuration that minimizes validation loss and improves prediction accuracy.

2.  **Additional Evaluation Metrics**: Beyond R² score, consider other metrics like Mean Absolute Error (MAE), Mean Squared Error (MSE), or Root Mean Squared Error (RMSE) to get a more comprehensive understanding of your model's performance.

3.  **Compare with other Models**: Implement and compare the performance of this LSTM model with other time series forecasting models (e.g., ARIMA, Prophet, or other neural network architectures).

4.  **Feature Engineering**: Incorporate more features into your dataset, such as trading volume, moving averages, or other technical indicators, to potentially improve prediction accuracy.

5.  **Ensemble Methods**: Combine multiple models to potentially achieve better performance than a single model alone.